# Dynamic Pricing Analysis: Online Retail II

## 1. Business Context
In competitive e-commerce markets, static pricing is no longer sufficient. Leading retailers like Amazon adjust prices millions of times per day based on demand, competition, and inventory levels. 

**Price Elasticity of Demand** measures how sensitive customers are to price changes. Understanding this helps in:
- Identifying products where price can be increased without losing much volume (Inelastic).
- Identifying products where small discounts can significantly boost volume (Elastic).

This project builds an end-to-end system to model elasticity and recommend optimal prices for the Online Retail II dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.preprocess import load_data, aggregate_product_data, create_price_demand_pairs, add_time_features
from src.elasticity import compute_price_elasticity, fit_demand_curve, plot_demand_curve
from src.pricing import find_optimal_price

sns.set_style('whitegrid')
%matplotlib inline

## 2. EDA
First, we load the data and look at the key distributions.

In [ ]:
df = load_data('data/online_retail.csv')
print(f"Dataset Size: {df.shape}")
df.head()

In [ ]:
# Revenue Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['Revenue'], bins=100, kde=True)
plt.title('Revenue Distribution (per transaction)')
plt.xlim(0, 100)
plt.show()

In [ ]:
# Top products by revenue
top_products = df.groupby(['StockCode', 'Description'])['Revenue'].sum().sort_values(ascending=False).head(10)
top_products.plot(kind='barh', title='Top 10 Products by Revenue')
plt.show()

## 3. Price Elasticity Analysis
We model demand sensitivity using log-log regression: $\ln(Q) = \alpha + \beta \ln(P) + \epsilon$, where $\beta$ is the price elasticity.

In [ ]:
price_demand = create_price_demand_pairs(df)
example_sc = top_products.index[0][0]
e, r2, interp, model = compute_price_elasticity(price_demand, example_sc)
print(f"Product: {example_sc}")
print(f"Elasticity: {e:.2f} ({interp})")
print(f"R-squared: {r2:.3f}")

## 4. Optimal Pricing
Using the fitted demand curve, we can find the price that maximizes revenue or profit.

In [ ]:
_, _, _, predict_func = fit_demand_curve(price_demand, example_sc)
cost = df[df['StockCode'] == example_sc]['Price'].mean() * 0.7
rev_p, prof_p, exp_rev, exp_prof = find_optimal_price(predict_func, cost)
print(f"Revenue Maximizing Price: £{rev_p:.2f}")
print(f"Profit Maximizing Price: £{prof_p:.2f}")

## 5. Conclusions
- The system successfully identifies price sensitivity across the portfolio.
- Dynamic factors (time, inventory) can be layered on top of the base elasticity model.
- High revenue products in the 'Inelastic' quadrant offer immediate opportunities for margin expansion.